In [ ]:
# import os
# import json
# from openai import OpenAI
# from dotenv import load_dotenv
# 
# load_dotenv()
# base_url = os.getenv("base_url3")
# key = os.getenv("key3")
# model = "deepseek-v4-flash"
# print(f"Model>>{model}")
# client = OpenAI(
#     api_key=key,
#     base_url=base_url
# )
# messages = [
#     {"role": "system",
#      "content": "知识库，收纳了很多小说的文本"
#      },
#     {"role": "user",
#      "content": "把三国演义小说文本流式输出，中间不要省略，换一章的时候，输出“=====================”加以区分段落"
#      }
# ]
# response = client.chat.completions.create(
#     model=model,
#     messages=messages,
#     max_tokens=50000,
#     temperature=1.5,
#     stream=True,
# )
# for chunk in response:
#     text = chunk.choices[0].delta.content
#     if chunk.choices[0] and (text is not None):
#         print(f"{text}", end="", flush=False)
# print(f"\n回答完毕！")

In [11]:
# ##sim0614>>仿真大模型自主调用工具执行函数
# import os
# import json
# from openai import OpenAI
# from dotenv import load_dotenv
# 
# load_dotenv()
# base_url = os.getenv("base_url3")
# key = os.getenv("key3")
# model = "deepseek-v4-flash"
# print(f"Model>>{model}")
# 
# def myfunc1(a,b):
#     # 计算两数之和
#     y=a+b
#     return y
# def myfunc2(n):
#     # 计算n的阶乘
#     y=1
#     for ind in range(1,n+1):
#         y=y*ind
#     return y
# def myfunc3():
#     print(f"啥也不用计算呢！")
# MyFuncs={
#     "myfunc1":{
#         "function":myfunc1,
#         "description":"a+b"
#     },
#     "myfunc2": {
#         "function": myfunc2,
#         "description": "计算阶乘"
#     },
#     "myfunc3":{
#         "function":myfunc3,
#         "description":"打印一行字符串！"
#     },
# }
# systemcontent="""
# 你是一个智能助手，现在我有三个函数可以调用，现在你要根据我的提问句子，自行判断调用哪个函数，传入什么参数。补充要求和信息如下：
# ### 意图识别
# (1)如果用户的问题里面含有方程，你要先解方程，然后再算
# ### 函数名称与功能
# (1)myfunc1(a,b):计算a,b两数的和返回结果
# (2)myfunc2(n):计算整数n的阶乘结果
# (3)myfunc3():打印一行文字
# ### 输出内容要求
# (1)你的输出内容要包含应该调用的函数的名字以及参数列表
# (2)你要根据自己的推理，识别出用户的意图，比如用户说计算五减去9,你应该判断出要调用函数myfunc1(a=5,b=-9)
# (3)你必须用标准的json格式给我输出回复信息,例如{"funcname":"myfunc1","args":[5,-9]}，不要带有Markdown的渲染符号哦
# """
# usercontent="""
# a=8,b=9,计算加法
# """
# 
# client=OpenAI(
#     base_url=base_url,
#     api_key=key,
# )
# res=client.chat.completions.create(
#     model=model,
#     messages=[
#         {"role":"system","content":systemcontent},
#         {"role":"user","content":usercontent},
#     ]
# )
# # var1=res.model_dump_json()# 先转换成json
# var1=res.to_dict()# 先转换成字典？
# var2=var1.get("choices","Error!")[0];
# var3=var2.get("message","No message!").get("content","No content!")
# print(f"var3>>\n{var3}")
# info=json.loads(var3)# 转换成字典格式
# print(f"data>>\n{info}")
# func_name=info.get('funcname')
# func_selected=MyFuncs.get(info.get("funcname","No function!")).get("function")# 获取函数对象
# args=info.get("args","No arg!")
# print(f"被调用的函数是：{func_name}\n")
# print(f"函数参数是：{args}\n")
# value=func_selected(*args)
# print(f"执行结果为：{value}")

Model>>deepseek-v4-flash
var3>>
{"funcname":"myfunc1","args":[8,9]}
data>>
{'funcname': 'myfunc1', 'args': [8, 9]}
被调用的函数是：myfunc1

函数参数是：[8, 9]

执行结果为：17


In [19]:
##sim0612>>仿真openai调用工具和函数
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
base_url = os.getenv("base_url3")
key = os.getenv("key3")
model = "deepseek-v4-flash"
print(f"Model>>{model}")
client = OpenAI(
    api_key=key,
    base_url=base_url,
)


# 1. 定义工具函数
def get_weather(city: str) -> str:
    """模拟查询天气"""
    weather_data = {
        "北京": "晴天,25°C,东南风3级",
        "上海": "阴天,22°C,东风2级",
        "广州": "多云,28°C,南风2级"
    }
    return weather_data.get(city, f"{city}:天气数据暂不可用")


def get_flight_price(origin: str, destination: str) -> str:
    """模拟查询机票价格"""
    if origin == "北京" and destination == "上海":
        return "经济舱 500元, 公务舱 1200元"
    elif origin == "上海" and destination == "广州":
        return "经济舱 450元, 公务舱 1000元"
    else:
        return f"{origin}到{destination}:暂未查询到航班信息"


def makePPT(fpath: str) -> str:
    print(f"完成ppt制作！")
    return (f"所有的ppt都已经搞定！")


# 2. 工具描述（告诉模型可以调用哪些函数）
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气情况",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_price",
            "description": "查询两个城市之间的机票价格",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "出发城市"},
                    "destination": {"type": "string", "description": "到达城市"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "makePPT",
            "description": "整理文档",
            "parameters": {
                "type": "object",
                "properties": {
                    "fpath": {"type": "string", "description": "文件路径"}
                },
                "required": ["fpath"]
            }
        }
    }
]

# 3. 工具分发映射
tool_map = {
    "get_weather": get_weather,
    "get_flight_price": get_flight_price,
    "makePPT": makePPT,
}


# 4. Agent主循环
def run_agent(user_input: str):
    messages = [
        {"role": "system", "content": "你是一个智能助手，可以根据用户问题调用工具获取信息。"},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    msg = response.choices[0].message
    # print(f"msg>>\n{msg}")
    print(f"=" * 35, "\n")
    # 无工具调用，直接返回
    if not msg.tool_calls:
        return msg.content

    # 执行工具调用并获取结果
    messages.append(msg)
    for tc in msg.tool_calls:
        print(f"当前tc>>\n{tc}")
        print(f"=" * 35)
        fn_name = tc.function.name  # 获取工具的名称
        fn_args = json.loads(tc.function.arguments)  # 获取调用参数，转换成字典形式
        result = tool_map[fn_name](**fn_args)
        print(f"-" * 35)
        print(f"调用工具：{fn_name}")
        print(f"传入参数：{fn_args}")
        print(f"返回结果：{result}")
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result  # 工具调用的返回结果添加到聊天记录里面
        })

    # 将工具结果交给模型生成最终回答
    final_response = client.chat.completions.create(
        model=model,
        messages=messages
    )  # 把工具返回的结果也带入到大模型对话里面，一起输入给大模型，重新返回回答
    return final_response.choices[0].message.content


# 5. 运行示例
if __name__ == "__main__":
    # # 测试1：查询天气
    # result1 = run_agent("北京今天天气怎么样？")
    # print(f"[最终回答] {result1}\n")
    #
    # # 测试2：查询机票价格
    # result2 = run_agent("从北京到上海的机票多少钱？")
    # print(f"[最终回答] {result2}\n")

    # 测试3：组合查询（可能同时调用两个工具）
    result3 = run_agent("我想从北京去上海去汇报一个电脑C盘的PPT，查一下北京的天气和机票价格")
    print(f"[最终回答] {result3}")

Model>>deepseek-v4-flash

当前tc>>
ChatCompletionMessageFunctionToolCall(id='call_00_Pl0iGdpeaP2ioilLv2fY6429', function=Function(arguments='{"city": "北京"}', name='get_weather'), type='function', index=0)
-----------------------------------
调用工具：get_weather
传入参数：{'city': '北京'}
返回结果：晴天,25°C,东南风3级
当前tc>>
ChatCompletionMessageFunctionToolCall(id='call_01_olnriMKSwz60sTvqa60z8224', function=Function(arguments='{"origin": "北京", "destination": "上海"}', name='get_flight_price'), type='function', index=1)
-----------------------------------
调用工具：get_flight_price
传入参数：{'origin': '北京', 'destination': '上海'}
返回结果：经济舱 500元, 公务舱 1200元
[最终回答] 好的，以下是目前查到的信息：

### ☀️ 北京天气
- **天气状况**：晴天
- **温度**：25°C
- **风力**：东南风3级

### ✈️ 北京→上海机票价格
- **经济舱**：**500元**
- **公务舱**：**1200元**

### 📊 关于电脑C盘的PPT
您提到要汇报一个电脑C盘的PPT，但我没有找到具体的文件信息。请问可以告诉我**C盘中的具体文件路径或文件名**吗？比如 `C:\xxx\xxx.pptx`，这样我才能帮您打开或处理这个PPT文件。
